# LangChain String Output Parser Reference

Developer-facing statements defined in `langchain_core.output_parsers.string`.

# `StrOutputParser: BaseTransformOutputParser[str]`

Extracts text content from model outputs as plain strings.

The parser performs no transformation on text values. Through `BaseTransformOutputParser`, it can process complete model outputs and streamed message or text chunks.

## Constructor

```python
StrOutputParser(
) -> None
```

The class defines no public configuration fields.

## Methods

### `is_lc_serializable`

Reports that the parser supports LangChain serialization.

```python
@classmethod
is_lc_serializable(
    cls,
) -> bool # Always True
```

### `get_lc_namespace`

Returns the LangChain serialization namespace.

```python
@classmethod
get_lc_namespace(
    cls,
) -> list[str] # `["langchain", "schema", "output_parser"]`
```

### `parse`

Returns the supplied text unchanged.

```python
@override
parse(
    self,
    text: str, # Text to return
) -> str # Unmodified input text
```

## Behaviour

When invoked with a string, the inherited runnable parsing flow returns that string unchanged.

When invoked with a message, the inherited parsing flow extracts the message text through its generation representation and passes it to `parse()`.

The inherited synchronous and asynchronous transform interfaces support streaming and emit string chunks as model output becomes available.

For serialization, the parser identifies its private serialization type as `"default"`.

In [ ]:
from collections.abc import AsyncIterator # Import the asynchronous iterator type

from langchain_core.messages import AIMessage, AIMessageChunk # Import real LangChain message classes
from langchain_core.output_parsers import StrOutputParser # Import the real string output parser


parser = StrOutputParser() # Create the string output parser

direct_result = parser.parse("Hello from LangChain") # Return the supplied text unchanged
print("Direct parse:", direct_result) # Display the direct parsing result

string_result = parser.invoke( # Parse a string through the runnable interface
    "Python is easy to learn.", # Provide plain text input
    config={"run_name": "parse_string"}, # Name the parser run
) # Finish invoking the parser

print("String invoke:", string_result) # Display the string result

message = AIMessage(content="This text came from an AIMessage.") # Create a real AI message
message_result = parser.invoke(message) # Extract the message text

print("Message invoke:", message_result) # Display the extracted message content

chunks = iter([ # Create synchronous streaming message chunks
    AIMessageChunk(content="Lang"), # Provide the first text chunk
    AIMessageChunk(content="Chain "), # Provide the second text chunk
    AIMessageChunk(content="streams output."), # Provide the final text chunk
]) # Finish creating the chunk iterator

print("\nSynchronous streaming:") # Display a heading

for parsed_chunk in parser.transform(chunks): # Parse each streamed chunk
    print(parsed_chunk, end="") # Display chunks without extra line breaks

print() # Move to the next output line


async def generate_chunks() -> AsyncIterator[AIMessageChunk]: # Define an asynchronous chunk source
    yield AIMessageChunk(content="Async ") # Yield the first asynchronous chunk
    yield AIMessageChunk(content="streaming ") # Yield the second asynchronous chunk
    yield AIMessageChunk(content="also works.") # Yield the final asynchronous chunk


print("\nAsynchronous streaming:") # Display a heading

async for parsed_chunk in parser.atransform(generate_chunks()): # Parse asynchronously in Jupyter
    print(parsed_chunk, end="") # Display each asynchronous chunk

print() # Move to the next output line

print("\nSerializable:", parser.is_lc_serializable()) # Display serialization support
print("LC namespace:", parser.get_lc_namespace()) # Display the serialization namespace